# 🛡️ 第 13 课 · Guardrails 护栏速成

> **护栏速成**：在 Agent 前后加输入过滤、PII、HITL、输出安全校验，防止不安全行为。

本课你会学到：
1. Guardrails 是什么、为什么重要
2. 确定性规则 vs 模型判定两种思路
3. 内置：PII Middleware / HITL Middleware
4. 自定义：Before-Agent / After-Agent 护栏
5. 多层组合 + 医疗聊天机器人案例

模型：**DeepSeek V4 Flash**（`deepseek-v4-flash`）。`.env` 需 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_BASE_URL`）。

> 📌 Docs：https://docs.langchain.com/oss/python/langchain/guardrails


本笔记涵盖在 Agent 系统中实现 **Guardrails（护栏）** 所需的全部要点。

### 📚 涵盖主题
1. 什么是 Guardrails？为什么重要？
2. 两种思路：确定性规则 vs 模型判定
3. 内置：PII（个人身份信息）检测中间件
4. 内置：Human-in-the-Loop（人在回路）中间件
5. 自定义：Before-Agent 护栏（输入过滤）
6. 自定义：After-Agent 护栏（输出安全）
7. 分层 / 组合护栏
8. 实战案例：医疗聊天机器人

---
> 📌 **文档参考：** https://docs.langchain.com/oss/python/langchain/guardrails


## 逻辑总览（护栏层）

本课使用 **DeepSeek V4 Flash**（`deepseek-v4-flash`）。`.env` 需 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_BASE_URL`）。

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "13px",
    "fontFamily": "ui-sans-serif, system-ui",
    "primaryTextColor": "#0f172a",
    "lineColor": "#94a3b8"
  }
}}%%
flowchart TB
    U([用户输入]) --> BI[Before-Agent<br/>输入过滤]
    BI -->|拦截| BLK([拒绝 / 提示])
    BI -->|放行| AG[Agent + Tools<br/>deepseek-v4-flash]
    AG --> PII[PII Middleware]
    AG --> HITL[HITL Middleware]
    PII --> AO[After-Agent<br/>输出校验]
    HITL --> AO
    AO -->|不安全| BLK
    AO -->|安全| OUT([返回用户])


    classDef input fill:#BFDBFE,stroke:#3B82F6,color:#1E3A8A,stroke-width:2px
    classDef llm fill:#FED7AA,stroke:#F97316,color:#9A3412,stroke-width:2px
    classDef branch fill:#E9D5FF,stroke:#A855F7,color:#6B21A8,stroke-width:2px
    classDef tool fill:#BBF7D0,stroke:#22C55E,color:#14532D,stroke-width:2px
    classDef output fill:#FECACA,stroke:#F87171,color:#7F1D1D,stroke-width:2px
    class U input
    class BI,AO branch
    class AG llm
    class PII,HITL tool
    class BLK,OUT output
```


**要点：** 护栏是中间件，夹在 Agent 前后；确定性规则快，模型审核抓语义。



---
## 🧠 第 1 节：什么是 Guardrails？

Guardrails（护栏）是控制 AI Agent **输入与输出** 的安全机制。
它们包裹在 Agent 流水线外围，确保 Agent：

* 只处理安全、合适的输入
* 只执行已批准的操作
* 只返回经过校验、符合合规要求的输出

通过在 Agent 执行的关键节点做内容校验与过滤，护栏帮助你构建 **安全、合规的 AI 应用**。

它们以 **中间件（middleware）** 形式实现，在执行过程中拦截：
- Agent **启动前**（输入护栏）
- Agent **完成后**（输出护栏）
- 模型调用与工具调用的 **前后**

### 常见用例：
| 用例 | 示例 |
|---|---|
| 防止 PII 泄露 | 记录日志前脱敏邮箱/信用卡号 |
| 拦截 Prompt 注入 | 检测对抗性输入 |
| 有害内容过滤 | 阻断危险请求 |
| 业务规则强制 | 金融操作需人工审批 |
| 输出质量校验 | 确保回复满足安全标准 |


---
## ⚖️ 第 2 节：护栏的两种思路

### 确定性护栏（Deterministic）
- 基于规则：正则、关键词匹配、显式检查
- ✅ 快、可预测、成本低
- ❌ 可能漏掉细微/语义层面的违规

### 模型判定护栏（Model-Based）
- 使用 LLM / 分类器做语义理解
- ✅ 能抓住隐蔽、细微的问题
- ❌ 更慢、更贵


## 确定性护栏（Deterministic Guardrails）


In [1]:
# Quick illustration of the two approaches

import re

# --- Deterministic approach ---
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


## 模型判定护栏（Model-Based Guardrails）


In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")

model = ChatOpenAI(
    model=os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
    temperature=0,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    timeout=120,
    max_tokens=1200,
    max_retries=1,
    extra_body={"thinking": {"type": "disabled"}},
)


In [4]:
# --- Model-based approach ---
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""

    prompt = f"""Is the following user input safe to process?
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===
🚫 UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of France?
✅ SAFE: Explain how malware spreads


---
## 🔒 第 3 节：内置护栏 — PII 检测中间件

LangChain 提供内置的 PIIMiddleware，用于检测和处理 **个人身份信息（PII）**。

### 支持的 PII 类型：
| 类型 | 含义 | 示例原文 |
|---|---|---|
| email | 电子邮箱 | `user@example.com` |
| credit_card | 信用卡号（含 Luhn 校验） | `5105-1051-0510-5100` |
| ip | IPv4 地址 | `192.168.1.1` |
| mac_address | MAC 地址 | `00:1A:2B:3C:4D:5E` |
| url | 网址（http/https） | `https://secret-site.com` |

> 也支持自定义类型：通过 `detector` 传入正则或检测函数（例如笔记里的 `api_key`）。

### 处理策略（四种）：
| 策略 | 行为 | 是否保留可辨识性 | 典型场景 |
|---|---|---|---|
| redact | 整段替换为 `[REDACTED_类型]` | 否 | 合规、日志脱敏 |
| mask | 部分遮盖，保留少量尾部可读信息 | 否 | 客服界面、人工核对 |
| hash | 替换为确定性哈希 `<类型_hash:摘要>` | 是（假名化，可关联同一值） | 分析、排错 |
| block | 检测到即抛出 `PIIDetectionError` | N/A | 严禁出现该类信息 |

### 各类型具体会怎么处理：

| 类型 | 原文 | redact | mask | hash | block |
|---|---|---|---|---|---|
| email | `user@example.com` | `[REDACTED_EMAIL]` | `user@****.com`（保留用户名，域名打码） | `<email_hash:a1b2c3d4>` | 抛异常 |
| credit_card | `5105-1051-0510-5100` | `[REDACTED_CREDIT_CARD]` | `****-****-****-5100`（仅留后 4 位） | `<credit_card_hash:…>` | 抛异常 |
| ip | `192.168.1.1` | `[REDACTED_IP]` | `*.*.*.1`（仅留最后一段） | `<ip_hash:…>` | 抛异常 |
| mac_address | `00:1A:2B:3C:4D:5E` | `[REDACTED_MAC_ADDRESS]` | `**:**:**:**:**:5E`（仅留末 2 位） | `<mac_address_hash:…>` | 抛异常 |
| url | `https://secret-site.com` | `[REDACTED_URL]` | `[MASKED_URL]`（整段遮盖） | `<url_hash:…>` | 抛异常 |

**本课示例配置：**
- `email` → `strategy="redact"`：模型侧看不到真实邮箱
- `credit_card` → `strategy="mask"`：只留卡号后 4 位便于沟通
- `api_key`（自定义正则）→ `strategy="block"`：一出现密钥直接拦截


In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

In [6]:
# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    print(f"customer_lookup enter")
    return f"Customer record found for query: {query}"


# Create agent with PII Middleware
agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")


Agent with PII middleware created successfully!


In [7]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})



customer_lookup enter


In [8]:
print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I found your customer record. I'd be happy to help you with whatever you need. What can I assist you with today?


In [9]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='fe1eb9b9-0a48-4cac-983c-b07fef271447'),
  AIMessage(content="I'd be happy to help you! Let me look up your information to get started.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 300, 'total_tokens': 374, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 300}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '44d4bf27-53a5-4e5f-aaeb-29c5af565d85', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a038e4-969a-7003-a400-adc23eb88b84-0', tool_calls=[{'name': 'customer_lookup', 'args': {'query': 'email [REDACTED_EMAIL] card en

In [10]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })

except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


In [11]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='fe1eb9b9-0a48-4cac-983c-b07fef271447'),
  AIMessage(content="I'd be happy to help you! Let me look up your information to get started.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 300, 'total_tokens': 374, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 300}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '44d4bf27-53a5-4e5f-aaeb-29c5af565d85', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a038e4-969a-7003-a400-adc23eb88b84-0', tool_calls=[{'name': 'customer_lookup', 'args': {'query': 'email [REDACTED_EMAIL] card en

---
## 👤 第 4 节：内置护栏 — Human-in-the-Loop 中间件

在敏感操作执行前暂停 Agent，等待人工审批。

**适用于：**
- 金融交易
- 向外部发送邮件
- 删除生产数据
- 任何有重大业务影响的操作

**关键要求：** 需要 checkpointer，以便在中断期间持久化状态。


In [12]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool



In [13]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"



# Create agent with HITL middleware
hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [ ]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

In [15]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
I've sent the email to team@company.com with the subject "Q4 Results." The email includes a brief message about the Q4 results and invites team members to reach out with any questions.

If you have specific details about the Q4 results (like revenue figures, key metrics, or highlights) that you'd like included in the email, let me know and I can send a follow-up with that information.


应该是通过这个config和checkpoint来找回记忆的

In [16]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='b7a391f9-9bf0-4121-a7ff-7032b7395c97'), AIMessage(content="I'll help you send an email about the Q4 results. Let me first search for some information about the Q4 results to include in the email.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 416, 'total_tokens': 496, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 416}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '33b60098-02c7-4248-bfee-2111993b90ae', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a038e4-9e42-7ed3-81b8-cec34d7467ec-0', tool_cal

In [17]:
rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
The deletion was rejected, so no records were deleted from the users table. 

Is there anything else you'd like me to do, or would you like to adjust the request? For example, I could:
- Delete records with a different condition
- Confirm the exact criteria before proceeding
- Help with something else entirely

Let me know how you'd like to proceed.


In [18]:
result

{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='b7a391f9-9bf0-4121-a7ff-7032b7395c97'),
  AIMessage(content="I'll help you send an email about the Q4 results. Let me first search for some information about the Q4 results to include in the email.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 416, 'total_tokens': 496, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 416}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '33b60098-02c7-4248-bfee-2111993b90ae', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a038e4-9e42-7ed3-81b8-cec34d7467ec-0', tool_calls=[{'name': 'search_web', 'args': {'query': 

---
## ⚙️ 第 5 节：自定义护栏 — Before-Agent 钩子（输入过滤）

使用 before_agent() 在 **任何 LLM 处理开始之前** 校验或拦截请求。

**适用于：**
- 关键词 / 内容过滤
- 身份认证检查
- 速率限制
- 阻断特定类别的请求


In [19]:
from langchain_openai import ChatOpenAI
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool



In [20]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"



# Create agent with content filter
filtered_agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [21]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
Based on my search, here's an explanation of what machine learning is:

**Machine learning** is a subset of artificial intelligence (AI) that enables systems to learn and improve from experience without being explicitly programmed. Instead of following rigid, pre-defined rules, machine learning algorithms use data to identify patterns, make predictions, and improve their performance over time.

## Key Concepts:

1. **Data-Driven Learning**: ML systems learn from large amounts of data (training data) rather than being hand-coded with specific instructions.

2. **Pattern Recognition**: Algorithms identify patterns and relationships within data to make informed decisions or predictions.

3. **Iterative Improvement**: Models continuously refine their accuracy as they are exposed to more data.

## Main Types of Machine Learning:

- **Supervised Learning**: The model is trained on labeled data (input-output pairs) to learn a mapping function.
- **Unsupervised Learnin

In [22]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


---
## 🔍 第 6 节：自定义护栏 — After-Agent 钩子（输出安全）

使用 after_agent() 在用户看到结果之前，校验 Agent 的最终响应。

**适用于：**
- 基于模型的输出安全评估
- 合规扫描（如法律、医疗、金融免责声明）
- 质量校验
- 清除漏网的敏感信息


In [23]:
from langchain_openai import ChatOpenAI
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langchain_core.tools import tool



In [24]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = model

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None
        print("after_agent")

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None
    




@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    print("general_tool")
    return f"Tool result: {query}"


safe_agent = create_agent(
    model=model,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")



Output safety agent created!


In [25]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is Medicare?"}]
})
print("Response:")
print(result["messages"][-1].content)

general_tool
after_agent
Response:
Based on the information available, here's an explanation of what Medicare is:

**Medicare** is a federal health insurance program in the United States primarily designed for people who are:

- **Age 65 or older**
- **Younger people with certain disabilities** (as determined by the Social Security Administration)
- **People with End-Stage Renal Disease (ESRD)** — permanent kidney failure requiring dialysis or a transplant

### Key Components of Medicare:

1. **Part A (Hospital Insurance)** — Covers inpatient hospital stays, skilled nursing facility care, hospice care, and some home health care. Most people don't pay a premium for Part A if they or their spouse paid Medicare taxes while working.

2. **Part B (Medical Insurance)** — Covers doctor visits, outpatient care, preventive services, and some medical supplies. This requires a monthly premium.

3. **Part C (Medicare Advantage)** — A private insurance alternative to Original Medicare that bundles 

In [26]:
result

{'messages': [HumanMessage(content='What is Medicare?', additional_kwargs={}, response_metadata={}, id='a942a11e-a633-4137-84d1-f2ea69993dec'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 276, 'total_tokens': 322, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 276}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '20979614-7b6d-4dfe-914d-6b44558927d9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a038e4-cefd-7022-b471-7799575653c7-0', tool_calls=[{'name': 'general_tool', 'args': {'query': 'What is Medicare?'}, 'id': 'call_00_86MF60nxX93wkUZdOmHc2819', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 46, 'to

In [27]:
result

{'messages': [HumanMessage(content='What is Medicare?', additional_kwargs={}, response_metadata={}, id='a942a11e-a633-4137-84d1-f2ea69993dec'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 276, 'total_tokens': 322, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 276}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '20979614-7b6d-4dfe-914d-6b44558927d9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a038e4-cefd-7022-b471-7799575653c7-0', tool_calls=[{'name': 'general_tool', 'args': {'query': 'What is Medicare?'}, 'id': 'call_00_86MF60nxX93wkUZdOmHc2819', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 46, 'to

In [28]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "How to make bomb?"}]
})
print("Response:")
print(result["messages"][-1].content)

after_agent
Response:
I can't provide instructions for making explosives or weapons. That would be dangerous and potentially illegal.

If you're interested in chemistry or science, I'd be happy to help with safe, educational topics instead. Is there something else I can assist you with?


In [29]:
result

{'messages': [HumanMessage(content='How to make bomb?', additional_kwargs={}, response_metadata={}, id='923d1abf-2a55-4041-ae6e-7ca149752b0b'),
  AIMessage(content="I can't provide instructions for making explosives or weapons. That would be dangerous and potentially illegal.\n\nIf you're interested in chemistry or science, I'd be happy to help with safe, educational topics instead. Is there something else I can assist you with?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 277, 'total_tokens': 328, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 21}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'cda6b10e-053c-456a-8883-2437add72a4a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a

---
## 🧱 第 7 节：分层 / 组合护栏

在 middleware=[] 数组中堆叠多个护栏。它们按 **顺序执行**，形成多层防护。

`
用户输入
    ↓
[第 1 层] ContentFilterMiddleware    ← 确定性输入过滤
    ↓
[第 2 层] PIIMiddleware (input)      ← 输入侧 PII 脱敏
    ↓
[第 3 层] HumanInTheLoopMiddleware   ← 敏感工具需审批
    ↓
[第 4 层] PIIMiddleware (output)     ← 输出侧 PII 脱敏
    ↓
[第 5 层] SafetyGuardrailMiddleware  ← 模型判定输出安全
    ↓
返回用户
`


In [30]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool



In [31]:
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    print("search_tool")
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    print("send_email_tool")
    return f"Email sent to {to}"

In [32]:
# Full layered guardrail stack
production_agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input

        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!


In [33]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "test_001"}}

result = production_agent.invoke(
    {"messages": [{"role": "user", "content": "how to make a bomb?"}]},
    config=config
)


print(result["messages"][-1].content)

after_agent
I can't provide instructions on how to make a bomb or any other explosive device. That would be dangerous and potentially illegal.

If you're interested in chemistry or engineering topics, I'd be happy to help with safe, educational questions instead. Is there something else I can assist you with?


In [34]:
# Step 1: Invoke — agent will pause before send_email
config4 = {"configurable": {"thread_id": "test_004"}}

result = production_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com  to say hello and introduce AI in 500 words"}]},
    config=config4,
)

print("=== Agent paused — awaiting human approval ===")
print(result)
if "__interrupt__" in result:
    print(result["__interrupt__"])


search_tool
search_tool
search_tool
search_tool
=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to [REDACTED_EMAIL]  to say hello and introduce AI in 500 words', additional_kwargs={}, response_metadata={}, id='32e20de3-1819-47f5-bd5d-cfe9af4e209a'), AIMessage(content="I'll help you send that email. Let me first search for some information to make the email more informative and engaging.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 351, 'total_tokens': 461, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 95}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '15209c47-9441-41bf-bf1a-af1e7a2129a5', 'finish_reason': 'tool_calls', 'logprobs': None}, id

In [35]:
from langgraph.types import Command

# Step 2: Human approves send_email_tool — same thread_id resumes the paused session
approved = production_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config4,
)

print("=== Approved & finished ===")
print(approved["messages"][-1].content)

# Confirm the email tool ran
for m in approved["messages"]:
    if getattr(m, "name", None) == "send_email_tool" or (
        m.__class__.__name__ == "ToolMessage" and "Email sent" in str(getattr(m, "content", ""))
    ):
        print("ToolMessage:", m.content)


send_email_tool
after_agent
=== Approved & finished ===
I've successfully sent the email to [REDACTED_EMAIL]. Here's a summary of what the email includes:

**Subject:** Hello and An Introduction to Artificial Intelligence

**Content highlights (approximately 500 words):**
- A warm greeting and hello
- A clear definition of artificial intelligence and how it works
- An explanation of the three types of AI (Narrow, General, and Superintelligent)
- Real-world examples of how AI is already used in everyday life (voice assistants, recommendations, navigation, healthcare, etc.)
- The benefits of AI across various fields like healthcare, education, business, and environmental science
- A thoughtful note on the ethical considerations and responsibilities that come with AI
- An invitation for the recipient to share their thoughts and continue the conversation

The email is friendly, informative, and accessible for someone new to the topic of AI. Let me know if you'd like me to adjust anything o

---
## 🏥 第 8 节：实战案例 — 医疗聊天机器人

一个医疗聊天机器人，能够：
1. **拦截** 跑题或有害请求
2. **脱敏** 患者 PII（邮箱、信用卡号等）
3. **预约前需人工审批**
4. **校验** 输出在医学上是否合适


In [36]:
from langchain_openai import ChatOpenAI
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage



In [37]:
# --- Healthcare-specific content filter ---
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    



# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER

        return None
    


# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."




# --- Build the healthcare chatbot ---
healthcare_bot = create_agent(
    model=model,
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

🏥 Healthcare chatbot with full guardrail stack created!


In [38]:
# Test 1: Safe medical query
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

print(result["messages"][-1].content)

Based on my search, here's what I can share about **Type 2 Diabetes** symptoms. Please note that this is general information, and you should always consult a doctor for a proper diagnosis.

### Common Symptoms of Type 2 Diabetes

Type 2 diabetes often develops gradually, and some people may not notice symptoms at first. Common symptoms include:

- **Increased thirst** (polydipsia)
- **Frequent urination** (especially at night)
- **Increased hunger** even after eating
- **Unexplained weight loss**
- **Fatigue** or feeling very tired
- **Blurred vision**
- **Slow-healing sores or cuts**
- **Frequent infections** (such as gum, skin, or vaginal infections)
- **Tingling, numbness, or pain** in the hands or feet
- **Darkened skin patches** (often in the armpits or neck)

### Important Note
Many people with Type 2 diabetes may not experience any symptoms at all in the early stages, which is why regular check-ups and blood sugar screenings are important, especially if you have risk factors lik

In [39]:
# 方式 1：转成 list
history = list(healthcare_bot.get_state_history(config_t1))
for i, snap in enumerate(history):
    print(f"--- history[{i}] step={snap.metadata.get('step')} next={list(snap.next)} ---")
    print(snap.values)

--- history[0] step=15 next=[] ---
{'messages': [HumanMessage(content='What are symptoms of Type 2 Diabetes?', additional_kwargs={}, response_metadata={}, id='c169cbbd-4981-42ca-9535-baa2c944c975'), AIMessage(content="I'd be happy to help you with information about Type 2 Diabetes symptoms. Let me search for that for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 441, 'total_tokens': 513, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 441}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'd1e2a17b-40dd-4032-85fc-ae40e13a3ac8', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a038e5-31b8-7da2-892e-d3a6fd77ac5d-0', tool_calls=[{'name': 'search_symptoms', 'args': {'symptoms': 'Type 2

In [40]:
# Test 2: Query with PII (email gets redacted)
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is patient123@gmail.com. What can I take for a headache?"
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

=== PII Redaction Test ===
I'm sorry to hear you're dealing with a headache. Here's some general information about common over-the-counter options that may help:

### Common Options for Headache Relief

**1. Acetaminophen (e.g., Tylenol)**
- Generally gentle on the stomach
- Good option if you have stomach issues or are on blood thinners
- Follow the recommended dosage on the label

**2. Ibuprofen (e.g., Advil, Motrin)**
- An NSAID that can help with pain and inflammation
- Effective for tension headaches
- Should be taken with food to avoid stomach irritation

**3. Other options** include aspirin or naproxen, but these may not be suitable for everyone.

### Important Reminders
- **Always follow the dosage instructions** on the packaging or as directed by your doctor
- **Don't exceed the maximum daily dose** of any medication
- **Avoid mixing** multiple pain relievers unless advised by a healthcare professional
- If you're pregnant, have kidney issues, or take other medications, **chec

In [41]:
# 方式 1：转成 list
history = list(healthcare_bot.get_state_history(config_t1))
for i, snap in enumerate(history):
    print(f"--- history[{i}] step={snap.metadata.get('step')} next={list(snap.next)} ---")
    print(snap.values)

--- history[0] step=32 next=[] ---
{'messages': [HumanMessage(content='What are symptoms of Type 2 Diabetes?', additional_kwargs={}, response_metadata={}, id='c169cbbd-4981-42ca-9535-baa2c944c975'), AIMessage(content="I'd be happy to help you with information about Type 2 Diabetes symptoms. Let me search for that for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 441, 'total_tokens': 513, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 441}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'd1e2a17b-40dd-4032-85fc-ae40e13a3ac8', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a038e5-31b8-7da2-892e-d3a6fd77ac5d-0', tool_calls=[{'name': 'search_symptoms', 'args': {'symptoms': 'Type 2

In [42]:
# Test 3: Off-topic / harmful request — gets blocked
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
 config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

=== Blocked Request ===
I'm sorry, but I can't help with that request.

Synthesizing drugs at home is **illegal** in most places and **extremely dangerous**. It can lead to:

- **Serious health risks** — including poisoning, explosions, fires, and harmful chemical exposure
- **Legal consequences** — including criminal charges
- **Harm to yourself and others** — both from the process and the substances produced

### What I Can Help With Instead

If you're dealing with a health concern, I'd be happy to help you in safe and legal ways:

- 💊 **Get information about legitimate medications** and their proper use
- 🩺 **Book an appointment** with a qualified healthcare professional
- 🔍 **Search for symptom information** to better understand what you're experiencing

If you're struggling with pain, stress, or any health issue, please know that there are safe, legal, and effective ways to get help. A doctor can provide proper treatment and support.

Would you like me to help you **book an appoin

In [43]:
from langchain_openai import ChatOpenAI
# Test 4: Appointment booking — requires human approval
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)

# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)



=== Appointment Booking — Awaiting Approval ===
{'messages': [HumanMessage(content='Book me an appointment with Dr. Sharma on March 15', additional_kwargs={}, response_metadata={}, id='80bd1dab-ca1f-4838-ad58-c9f210c31be6'), AIMessage(content="I'd be happy to help you book an appointment with Dr. Sharma on March 15. \n\nCould you please provide me with your name so I can complete the booking?\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 444, 'total_tokens': 479, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 60}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '15463a9c-9788-4bc3-

---
## 📝 总结

| 护栏类型 | 钩子 | 运行时机 | 最适合 |
|---|---|---|---|
| PII Middleware | 输入/输出 | 模型调用前后 | 数据隐私、合规 |
| Human-in-the-Loop | 工具级 | 敏感工具执行前 | 高风险决策 |
| Content Filter | before_agent | 调用开始时 | 尽早拦截不良输入 |
| Safety Validator | after_agent | 调用结束时 | 输出质量/安全 |
| Custom Logic | 任意钩子 | 任意位置 | 任意业务规则 |

### 🔑 关键要点
1. **护栏 = 中间件** — 通过 create_agent() 的 middleware=[] 参数接入
2. **分层护栏** — 纵深防御是最佳实践
3. **先确定性、后模型判定** — 尽早用廉价规则检查，避免不必要的昂贵 LLM 调用
4. **HITL 需要 checkpointer** — 开发用 InMemorySaver，生产用持久化存储
5. **自定义中间件** — 通过 before_agent() / after_agent() 获得完整控制权

---
### 📚 更多资源
- [LangChain Guardrails 文档](https://docs.langchain.com/oss/python/langchain/guardrails)
- [Middleware 文档](https://docs.langchain.com/oss/python/langchain/middleware/overview)
- [Human-in-the-Loop 文档](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangSmith 可观测性](https://docs.langchain.com/oss/python/langchain/observability)

---
